## **Ejercicio 8. Nodos centrales y participantes puente**

---

## **Reconstrucción de la red**

In [1]:
from pathlib import Path

import networkx as nx
import pandas as pd

SEMILLA = 123

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 170)

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DIR_PROCESSED = RAIZ / "data" / "processed"

comentarios = pd.read_csv(DIR_PROCESSED / "comentarios_limpio.csv", keep_default_na=False)
videos = pd.read_csv(DIR_PROCESSED / "videos_limpio.csv", keep_default_na=False)
comentarios["me_gusta"] = pd.to_numeric(comentarios.me_gusta)
videos["vistas"] = pd.to_numeric(videos.vistas)

titulo = videos.set_index("video_id").title.to_dict()
canal_por_video = videos.set_index("video_id").channel_name.to_dict()
nombre_autor = comentarios.set_index("author_channel_id").author_name.to_dict()

bipartita = nx.Graph()
bipartita.add_nodes_from(comentarios.author_channel_id.unique(), tipo="autor", bipartite=0)
bipartita.add_nodes_from(comentarios.video_id.unique(), tipo="video", bipartite=1)
for (autor, video), n in comentarios.groupby(["author_channel_id", "video_id"]).size().items():
    bipartita.add_edge(autor, video, peso=int(n))

conjunto_autores = {n for n, d in bipartita.nodes(data=True) if d["tipo"] == "autor"}
conjunto_videos = {n for n, d in bipartita.nodes(data=True) if d["tipo"] == "video"}

print(f"bipartita: {bipartita.number_of_nodes()} nodos ({len(conjunto_autores)} autores, "
      f"{len(conjunto_videos)} videos), {bipartita.number_of_edges()} aristas")

bipartita: 351 nodos (332 autores, 19 videos), 343 aristas


---

## **8.1. Medidas de centralidad**

Se calculan cuatro medidas, cada una responde una pregunta simple:

- **Grado**: ¿en cuántos videos distintos comentó un autor? o ¿cuántos autores distintos comentaron
  un video?
- **Grado ponderado**: igual, pero contando también cuántas veces repitió.
- **Intermediación**: ¿qué tan seguido un nodo es el "puente" entre otros dos? Aquí sirve para
  encontrar a quienes conectan grupos que si no, quedarían separados.
- **PageRank**: parecido al grado, pero da más peso si tus conexiones son con nodos importantes.

No se usa cercanía porque la red tiene 10 partes desconectadas entre sí, y esa medida solo tiene
sentido dentro de una misma parte.

In [2]:
grado = dict(bipartita.degree())
grado_pond = dict(bipartita.degree(weight="peso"))
intermediacion = nx.betweenness_centrality(bipartita, normalized=True)
pagerank = nx.pagerank(bipartita, weight="peso")

centralidad = pd.DataFrame({
    "grado": grado,
    "grado_ponderado": grado_pond,
    "intermediacion": intermediacion,
    "pagerank": pagerank,
}).round(5)
centralidad["tipo"] = ["autor" if n in conjunto_autores else "video" for n in centralidad.index]
centralidad.shape

(351, 5)

---

## **8.2. Interpretación por separado: autores y videos**

### **Autores: recurrencia y diversidad de participación**

In [3]:
tabla_autores = centralidad[centralidad.tipo == "autor"].drop(columns="tipo").copy()
tabla_autores.insert(0, "autor", [nombre_autor[a] for a in tabla_autores.index])
tabla_autores.sort_values("intermediacion", ascending=False).head(10)

,autor,grado,grado_ponderado,intermediacion,pagerank
UCHTGCgY2l-DQJIvlA_cpa_Q,@virgiliogarcia3039,2,3,0.23158,0.00304
UCpsKOkt5iWTbenzeuq7dsmQ,@inge_vergueta,3,3,0.21808,0.00332
UCzZ6dDCEsLMXbc5pCFXIprw,@josegil3813,2,2,0.17683,0.00208
UCylqlpsh8ENNM1LqgQ1KbxA,@Jel.Awesh.M,2,3,0.08915,0.00346
UCvcu1kR8xMYy_I1Ty-2mV2Q,@hashojea7348,3,4,0.05971,0.00454
UCjZgFEowMBrsjodqfovzdKw,@franciscoflores3120,2,2,0.05790,0.00229
UCbrtvygfRT6QXqWDNrOf-cA,@Alejandro00710,2,2,0.03230,0.00245
UCsUCN0Yq_UyTvKjOJLmNrnQ,@moisesvaldez4043,2,2,0.03186,0.00248
UCdFlugHJJa4l3YqWuNRmvXw,@MarcosCarillo-b1r,2,2,0.03186,0.00270
UCp_dAUfv8i3LW3zWc24dLaQ,@MarioCortez-o6v,1,1,0.00000,0.00128


De 332 autores, solo 9 comentaron en más de un video; el resto (97.3 %) comentó en uno solo y por
eso su intermediación es casi cero. De esos 9, @inge_vergueta y @virgiliogarcia3039 destacan porque
sus videos pertenecen a grupos distintos: son el único hilo que conecta esos grupos entre sí.

### **Videos: alcance y capacidad de conectar audiencias**

In [4]:
tabla_videos = centralidad[centralidad.tipo == "video"].drop(columns="tipo").copy()
tabla_videos.insert(0, "video", [titulo[v] for v in tabla_videos.index])
tabla_videos.insert(1, "canal", [canal_por_video[v] for v in tabla_videos.index])
tabla_videos.sort_values("pagerank", ascending=False).head(10)

,video,canal,grado,grado_ponderado,intermediacion,pagerank
n8iP75gIpmw,Qué rico come tu diputado,Quorum,128,161,0.56570,0.16729
j43HgwYFKfk,La cooptación de Walter Mazariegos en la USAC,Quorum,49,50,0.22747,0.06319
6W4u8sGEnGM,Inician los trabajos de recuperación del Puent...,Gobierno de la República de Guatemala,32,45,0.18762,0.04259
lj983NWyAQY,Plan 2032 Ciudad de Guatemala,Municipalidad de Guatemala,25,25,0.00491,0.03427
OkXlHx0hx-8,EE.UU. envía a mexicanos deportados a Guatemal...,Noticias Telemundo,18,25,0.00251,0.02511
PjmxCj-a9Hg,Conferencia de Prensa del Gobierno de Guatemal...,Gobierno de la República de Guatemala,19,25,0.24403,0.02490
yLZS3JiEBg8,Arroz con pollo a la MONOPOLIO,Quorum,16,16,0.06367,0.02045
06mFNPU0aB8,Capturan a presuntos delincuentes disfrazados ...,Noti7,13,14,0.05472,0.01751
ndAZjHqzzT8,Internet: escoger el menos malo,Quorum,10,12,0.12829,0.01328
is3Mk5oC19g,Capturan a ladrón que había quedado grabado mi...,TN23 Guatemala,7,7,0.02765,0.00986


In [5]:
# Las afirmaciones del texto se calculan aqui para que sean auditables.
solo_videos = centralidad[centralidad.tipo == "video"].copy()
solo_videos["vistas"] = [videos.set_index("video_id").vistas[v] for v in solo_videos.index]

print("CORRELACIONES ENTRE MEDIDAS, SOLO NODOS DE VIDEO")
print(f"  grado vs pagerank  -> Pearson {solo_videos.grado.corr(solo_videos.pagerank):.4f}, "
      f"Spearman {solo_videos.grado.corr(solo_videos.pagerank, method='spearman'):.4f}")
print(f"  vistas vs grado    -> Pearson {solo_videos.vistas.corr(solo_videos.grado):.4f}, "
      f"Spearman {solo_videos.vistas.corr(solo_videos.grado, method='spearman'):.4f}")
print()
print("  el video que rompe la relacion lineal entre vistas y comentaristas:")
print(solo_videos.nlargest(1, "vistas")[["grado", "vistas"]].to_string())

CORRELACIONES ENTRE MEDIDAS, SOLO NODOS DE VIDEO


  grado vs pagerank  -> Pearson 0.9998, Spearman 0.9978
  vistas vs grado    -> Pearson 0.0995, Spearman 0.8118

  el video que rompe la relacion lineal entre vistas y comentaristas:
             grado  vistas
lj983NWyAQY     25  304089


En los videos, grado y PageRank van casi de la mano, con una correlación de Spearman de 0.998: el
video con más comentaristas, *Qué rico come tu diputado* con 128 autores, también es el más
importante en PageRank. La intermediación sí marca una diferencia, ya que *Conferencia de Prensa del
Gobierno* tiene menos autores que *Arroz con pollo a la MONOPOLIO* pero conecta mejor a grupos
distintos.

La relación entre vistas y número de comentaristas depende de cómo se mida, y por eso conviene
reportar las dos correlaciones. En rangos la asociación es fuerte, con un Spearman de 0.812, es
decir, los videos más vistos tienden a ocupar también las primeras posiciones en comentaristas. En
magnitud, en cambio, la asociación desaparece, con un Pearson de apenas 0.0995, y el responsable es
*Plan 2032 Ciudad de Guatemala*: acumula 304,089 vistas, doce veces más que cualquier otro video con
participación, pero solo reúne 25 comentaristas. Dicho esto, la visibilidad ordena pero no escala la
participación.

---

## **8.3. Participantes recurrentes, autores puente y videos articuladores**

In [6]:
componente_mayor = bipartita.subgraph(max(nx.connected_components(bipartita), key=len))
articulacion = set(nx.articulation_points(componente_mayor))
autores_articulacion = sorted((a for a in articulacion if a in conjunto_autores),
                               key=lambda a: intermediacion[a], reverse=True)
videos_articulacion = sorted((v for v in articulacion if v in conjunto_videos),
                              key=lambda v: intermediacion[v], reverse=True)

print(f"Autores recurrentes (grado > 1): {sum(1 for a in conjunto_autores if grado[a] > 1)} de {len(conjunto_autores)}")
print(f"Puntos de articulacion en la componente mayor: {len(articulacion)} "
      f"({len(autores_articulacion)} autores, {len(videos_articulacion)} videos)")
print()
print("AUTORES PUENTE (su salida fragmenta la componente mayor)")
for a in autores_articulacion:
    vecinos = ", ".join(titulo[v][:28] for v in bipartita.neighbors(a))
    print(f"  {nombre_autor[a]:22s} grado={grado[a]}  inter={intermediacion[a]:.3f}  videos: {vecinos}")
print()
print("VIDEOS ARTICULADORES (su salida fragmenta la componente mayor)")
for v in videos_articulacion:
    print(f"  {titulo[v][:42]:44s} grado={grado[v]:3d}  inter={intermediacion[v]:.3f}  canal: {canal_por_video[v]}")

Autores recurrentes (grado > 1): 9 de 332
Puntos de articulacion en la componente mayor: 17 (7 autores, 10 videos)

AUTORES PUENTE (su salida fragmenta la componente mayor)
  @virgiliogarcia3039    grado=2  inter=0.232  videos: Conferencia de Prensa del Go, Qué rico come tu diputado
  @inge_vergueta         grado=3  inter=0.218  videos: La cooptación de Walter Maza, Qué rico come tu diputado, Internet: escoger el menos m
  @josegil3813           grado=2  inter=0.177  videos: Inician los trabajos de recu, Conferencia de Prensa del Go
  @hashojea7348          grado=3  inter=0.060  videos: Caminar en una ciudad hecha , Internet: escoger el menos m, Arroz con pollo a la MONOPOL
  @franciscoflores3120   grado=2  inter=0.058  videos: Capturan a presuntos delincu, Inician los trabajos de recu
  @moisesvaldez4043      grado=2  inter=0.032  videos: Bloqueos en Guatemala este 3, Qué rico come tu diputado
  @MarcosCarillo-b1r     grado=2  inter=0.032  videos: Capturan a ladrón que había , La coop

**Recurrentes**: los mismos 9 autores que comentaron en más de un video.

**Autores puente**: 7 de esos 9 son "puntos de articulación", es decir, si se van, su video pequeño
queda desconectado del resto de la red.

**Videos articuladores**: 10 videos cumplen el mismo papel, desde el más grande (*Qué rico come tu
diputado*, que sostiene a decenas de autores exclusivos) hasta videos chicos que son la única puerta
de entrada de su grupo de comentaristas.

En resumen: la red no tiene un núcleo fuerte, depende de pocos nodos (7 autores y 10 videos) que si
desaparecieran, la partirían en pedazos.